## DDL Gold: pf.gold.dim_licencia  (SCD Type 1)
## Dimension de licencias (tag license:*) extraida en Silver.

In [0]:
%sql
DROP TABLE IF EXISTS pf.gold.dim_licencia;

CREATE TABLE IF NOT EXISTS pf.gold.dim_licencia (
    licencia_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) COMMENT 'PK - Surrogate Key',
    license_tag STRING NOT NULL COMMENT 'BK - tag license:*',
    descripcion STRING COMMENT 'Descripcion',
    es_permissiva BOOLEAN COMMENT 'True si Apache/MIT (open-weight)',
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP COMMENT 'UTC',
    PRIMARY KEY (licencia_id),
    CONSTRAINT uniq_dim_licencia UNIQUE (license_tag)
)
USING DELTA
TBLPROPERTIES (
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults = 'supported'
)
COMMENT 'Dimension Licencia - SCD Type 1';

In [0]:
%sql
-- Carga/actualizacion SCD1. NULL se trata como 'sin_licencia'.
MERGE INTO pf.gold.dim_licencia AS t
USING (
    SELECT COALESCE(license_tag, 'sin_licencia') AS license_tag,
           SUM(1) AS n_models
    FROM pf.silver.modelos
    GROUP BY COALESCE(license_tag, 'sin_licencia')
) AS s
ON t.license_tag = s.license_tag
WHEN MATCHED THEN
    UPDATE SET t.descripcion = CASE
                    WHEN s.license_tag IN ('license:apache-2.0','license:mit') THEN 'Open-weight permisiva'
                    WHEN s.license_tag = 'sin_licencia' THEN 'Sin licencia declarada'
                    ELSE 'Licencia de uso restringido'
                END,
               t.es_permissiva = (s.license_tag IN ('license:apache-2.0','license:mit'))
WHEN NOT MATCHED THEN
    INSERT (license_tag, descripcion, es_permissiva, _createdAt)
    VALUES (s.license_tag,
            CASE WHEN s.license_tag IN ('license:apache-2.0','license:mit') THEN 'Open-weight permisiva'
                 WHEN s.license_tag = 'sin_licencia' THEN 'Sin licencia declarada'
                 ELSE 'Licencia de uso restringido'
            END,
            (s.license_tag IN ('license:apache-2.0','license:mit')),
            CURRENT_TIMESTAMP());

In [0]:
%sql

SELECT licencia_id, license_tag, descripcion, es_permissiva FROM pf.gold.dim_licencia ORDER BY license_tag;